# 04: Évaluation MedGemma-4B sur RSNA dev (Google Colab)

Ce notebook fait tourner **MedGemma-4B-IT** (le vrai VLM) sur les 150 cas dev RSNA
et compare les résultats au fallback règles.

| | Baseline règles | MedGemma baseline | MedGemma improved |
|---|---|---|---|
| Modèle | Score opacité focale | google/medgemma-4b-it | google/medgemma-4b-it |
| Prompt |: | baseline_prompt.txt | improved_prompt.txt |
| Accuracy (règles) | 35.3% | **?** | **?** |
| Macro-F1 (règles) | 27.6% | **?** | **?** |

> **Prérequis** : GPU T4 activé dans Colab (Exécution → Modifier le type d'exécution → GPU T4)
> **Données** : dossier du projet uploadé sur Google Drive

## Étape 1: GPU check + installation

In [ ]:
# Vérifier que le GPU est bien disponible
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU disponible :')
    print(result.stdout[:500])
else:
    print('ATTENTION : Aucun GPU détecté.')
    print('Va dans Exécution → Modifier le type exécution → GPU T4, puis relance.')

In [ ]:
%%capture
!pip install transformers>=4.41.0 accelerate>=0.30.0 bitsandbytes>=0.43.0 \
             huggingface_hub pillow opencv-python-headless python-dotenv

## Étape 2: Monter Google Drive + charger le projet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

# MODIFIE ce chemin si ton dossier est ailleurs dans Drive
PROJECT_ROOT = Path('/content/drive/MyDrive/assistant-radiologue-virtuel-main')

if not PROJECT_ROOT.exists():
    print(f'ERREUR : dossier introuvable -> {PROJECT_ROOT}')
    print('Vérifie que tu as bien uploadé le dossier du projet dans ton Drive.')
else:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f'Projet chargé : {PROJECT_ROOT}')
    print(f'Images RSNA   : {len(list((PROJECT_ROOT / "data/rsna/processed/images").glob("*.png")))} images')

## Étape 3: Authentification HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Méthode recommandée : stocker le token dans les secrets Colab
# (icône clé dans la barre gauche → ajouter HF_TOKEN)
try:
    hf_token = userdata.get('HF_TOKEN')
    print('Token chargé depuis les secrets Colab.')
except Exception:
    # Fallback : charger depuis .env du projet
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')
    hf_token = os.getenv('HF_TOKEN')
    print('Token chargé depuis .env')

if not hf_token:
    raise ValueError('HF_TOKEN introuvable. Ajoute-le dans les secrets Colab.')

login(token=hf_token, add_to_git_credential=False)
print('Authentification HuggingFace OK')

## Étape 4: Charger MedGemma-4B

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = 'google/medgemma-4b-it'
print(f'Chargement de {MODEL_ID} en float16...')
print(f'GPU : {torch.cuda.get_device_name(0)}: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    token=hf_token,
)
model.eval()
print('MedGemma chargé avec succès !')

## Étape 5: Fonction d'inférence MedGemma

In [ ]:
import json, re, time
from PIL import Image

ALLOWED_CLASSES = {'normal', 'suspected_opacity', 'uncertain'}
WARNING = 'Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.'

def read_prompt(name):
    p = PROJECT_ROOT / 'prompts' / name
    return p.read_text(encoding='utf-8') if p.exists() else ''

def medgemma_predict(image_path, prompt_file, prompt_version):
    t0 = time.perf_counter()
    image = Image.open(image_path).convert('RGB')
    prompt_text = read_prompt(prompt_file)

    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': prompt_text},
        ]
    }]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt'
    ).to(model.device, dtype=torch.float16)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=400, do_sample=False)

    input_len = inputs['input_ids'].shape[-1]
    raw = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)

    # Parser le JSON dans la réponse
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        data = json.loads(match.group(0))
    else:
        data = {}

    label = data.get('predicted_class', 'uncertain')
    if label not in ALLOWED_CLASSES:
        label = 'uncertain'

    try:
        confidence = float(data.get('confidence', 0.5))
        confidence = max(0.0, min(1.0, confidence))
    except (TypeError, ValueError):
        confidence = 0.5

    return {
        'predicted_class'  : label,
        'confidence'       : round(confidence, 3),
        'image_quality'    : data.get('image_quality', 'unknown'),
        'visual_evidence'  : data.get('visual_evidence', []),
        'justification'    : data.get('justification', raw[:200]),
        'limitations'      : data.get('limitations', []),
        'warning'          : WARNING,
        'model_name'       : MODEL_ID,
        'prompt_version'   : prompt_version,
        'latency_ms'       : int((time.perf_counter() - t0) * 1000),
        'raw_output'       : raw,
    }

print('Fonction medgemma_predict définie.')

## Étape 6: Test rapide sur 1 image

In [ ]:
import csv
from IPython.display import display

cases_csv = PROJECT_ROOT / 'data' / 'rsna' / 'cases.csv'
with open(cases_csv, newline='', encoding='utf-8') as f:
    dev_cases = [r for r in csv.DictReader(f) if r['split'] == 'dev']

print(f'{len(dev_cases)} cas dev chargés.')

# Test sur 1 image
test_case = dev_cases[0]
img_path = PROJECT_ROOT / test_case['image_path']

print(f'Test sur : {test_case["case_id"]}')
print(f'Label réel : {test_case["label"]}')

display(Image.open(img_path).resize((256, 256)))

pred = medgemma_predict(img_path, 'baseline_prompt.txt', 'baseline_v1')
print(f'Prédit    : {pred["predicted_class"]}  (confiance {pred["confidence"]})')
print(f'Latence   : {pred["latency_ms"]} ms')
print(f'Correct   : {pred["predicted_class"] == test_case["label"]}')
print(f'\nRéponse brute :\n{pred["raw_output"][:500]}')

## Étape 7: Évaluation complète (150 cas dev)

In [ ]:
from collections import Counter
import numpy as np

def macro_f1(y_true, y_pred):
    classes = list(ALLOWED_CLASSES)
    f1s = []
    for c in classes:
        tp = sum(t == c and p == c for t, p in zip(y_true, y_pred))
        fp = sum(t != c and p == c for t, p in zip(y_true, y_pred))
        fn = sum(t == c and p != c for t, p in zip(y_true, y_pred))
        prec = tp / (tp + fp) if (tp + fp) else 0
        rec  = tp / (tp + fn) if (tp + fn) else 0
        f1s.append(2 * prec * rec / (prec + rec) if (prec + rec) else 0)
    return round(sum(f1s) / len(f1s), 4)

results_base = []
results_imp  = []

print('=== Baseline MedGemma ===')
for i, case in enumerate(dev_cases):
    img_path = PROJECT_ROOT / case['image_path']
    pred = medgemma_predict(img_path, 'baseline_prompt.txt', 'baseline_v1')
    results_base.append({
        'case_id': case['case_id'], 'label': case['label'],
        'predicted_class': pred['predicted_class'],
        'confidence': pred['confidence'], 'latency_ms': pred['latency_ms'],
        'correct': case['label'] == pred['predicted_class'],
    })
    if (i + 1) % 25 == 0:
        done = results_base
        acc = sum(r['correct'] for r in done) / len(done)
        print(f'  [{i+1}/150]  acc provisoire={acc:.2%}  dist={dict(Counter(r["predicted_class"] for r in done))}')

print('\n=== Improved MedGemma ===')
for i, case in enumerate(dev_cases):
    img_path = PROJECT_ROOT / case['image_path']
    pred = medgemma_predict(img_path, 'improved_prompt.txt', 'improved_v1')
    results_imp.append({
        'case_id': case['case_id'], 'label': case['label'],
        'predicted_class': pred['predicted_class'],
        'confidence': pred['confidence'], 'latency_ms': pred['latency_ms'],
        'correct': case['label'] == pred['predicted_class'],
    })
    if (i + 1) % 25 == 0:
        done = results_imp
        acc = sum(r['correct'] for r in done) / len(done)
        print(f'  [{i+1}/150]  acc provisoire={acc:.2%}  dist={dict(Counter(r["predicted_class"] for r in done))}')

## Étape 8: Métriques et comparaison

In [ ]:
import pandas as pd

def compute_metrics(results, name):
    y_t = [r['label'] for r in results]
    y_p = [r['predicted_class'] for r in results]
    n = len(y_t)
    acc  = sum(t == p for t, p in zip(y_t, y_p)) / n
    f1   = macro_f1(y_t, y_p)
    tp_o = sum(t == 'suspected_opacity' and p == 'suspected_opacity' for t, p in zip(y_t, y_p))
    fn_o = sum(t == 'suspected_opacity' and p != 'suspected_opacity' for t, p in zip(y_t, y_p))
    sens = tp_o / (tp_o + fn_o) if (tp_o + fn_o) else 0
    tn_n = sum(t == 'normal' and p == 'normal' for t, p in zip(y_t, y_p))
    fp_n = sum(t == 'normal' and p != 'normal' for t, p in zip(y_t, y_p))
    spec = tn_n / (tn_n + fp_n) if (tn_n + fp_n) else 0
    ur   = sum(p == 'uncertain' for p in y_p) / n
    lat  = float(np.median([r['latency_ms'] for r in results]))
    return {'variante': name, 'accuracy': round(acc, 4), 'macro_f1': f1,
            'sensitivity_opacity': round(sens, 4), 'specificity_normal': round(spec, 4),
            'uncertain_rate': round(ur, 4), 'median_latency_ms': lat}

m_base_rules = {'variante': 'baseline_rules',  'accuracy': 0.3533, 'macro_f1': 0.2756,
                'sensitivity_opacity': 0.36, 'specificity_normal': 0.70,
                'uncertain_rate': 0.0, 'median_latency_ms': 22}
m_imp_rules  = {'variante': 'improved_rules',  'accuracy': 0.4400, 'macro_f1': 0.4139,
                'sensitivity_opacity': 0.18, 'specificity_normal': 0.48,
                'uncertain_rate': 0.5267, 'median_latency_ms': 21}

m_base_mg = compute_metrics(results_base, 'baseline_medgemma')
m_imp_mg  = compute_metrics(results_imp,  'improved_medgemma')

df = pd.DataFrame([m_base_rules, m_imp_rules, m_base_mg, m_imp_mg]).set_index('variante')
print('=== COMPARAISON COMPLÈTE ===')
print(df[['accuracy', 'macro_f1', 'sensitivity_opacity', 'specificity_normal', 'uncertain_rate']].to_string())

## Étape 9: Sauvegarde des résultats

In [ ]:
OUT = PROJECT_ROOT / 'eval' / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

# CSVs prédictions MedGemma
for fname, rows in [
    ('rsna_dev_baseline_medgemma_predictions.csv', results_base),
    ('rsna_dev_improved_medgemma_predictions.csv', results_imp),
]:
    with open(OUT / fname, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)
    print(f'Sauvegardé : {fname}')

# JSON métriques complètes
all_metrics = [m_base_rules, m_imp_rules, m_base_mg, m_imp_mg]
with open(OUT / 'rsna_dev_full_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=2, ensure_ascii=False)
print('Sauvegardé : rsna_dev_full_comparison.json')

print('\n=== RÉSUMÉ FINAL ===')
print(f'Baseline règles  : acc={m_base_rules["accuracy"]:.1%}  F1={m_base_rules["macro_f1"]:.1%}')
print(f'Improved règles  : acc={m_imp_rules["accuracy"]:.1%}  F1={m_imp_rules["macro_f1"]:.1%}')
print(f'Baseline MedGemma: acc={m_base_mg["accuracy"]:.1%}  F1={m_base_mg["macro_f1"]:.1%}')
print(f'Improved MedGemma: acc={m_imp_mg["accuracy"]:.1%}  F1={m_imp_mg["macro_f1"]:.1%}')